In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import json
from collections import Counter, defaultdict
from pathlib import Path
from zipfile import ZipFile, ZIP_STORED

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
import numpy as np
from contextlib import ExitStack
from io import BytesIO
from PIL import Image

In [3]:
DRIVE = Path('/content/drive/MyDrive/WasteScope')
ARCHIVE_DIR = DRIVE / 'raw_archives'
TASKS = {
    'detection': ARCHIVE_DIR / 'detection_task.zip',
    'tracking': ARCHIVE_DIR / 'tracking_task.zip',
    'segmentation': ARCHIVE_DIR / 'segmentation_task.zip'
}
CLASSES = [
    'container_yellow', 'container_green', 'container_default',
    'container_blue', 'container_oil', 'container_battery',
    'container_biodegradable'
]

OUTPUT = DRIVE / 'prepared' / 'detection_7class_v2'
MANIFEST = DRIVE / 'manifests' / 'video_split_7class_v2.json'
SEED = 42
RATIOS = np.array([0.70, 0.15, 0.15])
N_TRIALS = 1000
SPLIT_NAMES = ('train', 'val', 'test')

In [4]:
with ZipFile(TASKS['segmentation']) as archive:
    json_name = next(name for name in archive.namelist() if Path(name).name == 'labels.json')
    coco = json.loads(archive.read(json_name).decode('utf-8-sig'))
    
categories = {item['id']: item['name'] for item in coco['categories']}
overflow_id = next(cid for cid, name in categories.items() if name.lower() == 'overflow')
first_container = min(cid for cid, name in categories.items() if name.startswith('container_'))

In [5]:
DETECTION_MAP = {
    cid - 1: CLASSES.index(name) for cid, name in categories.items() if name in CLASSES
}
TRACKING_MAP = {
    cid - first_container: CLASSES.index(name)
    for cid, name in categories.items() if name in CLASSES
}

DETECTION_SKIP = {cid - 1 for cid, name in categories.items() if name not in CLASSES}
TRACKING_SKIP = {
    cid - first_container for cid, name in categories.items()
    if name.startswith('container_') and name not in CLASSES
}

In [6]:
images, labels = {}, {}

for task, path in TASKS.items():
    with ZipFile(path) as archive:
        image_names = sorted(name for name in archive.namelist() if name.lower().endswith('.png'))
        label_names = sorted(name for name in archive.namelist() if name.lower().endswith('.txt') and Path(name).parent.name == 'labels')
        
    images[task] = {Path(name).stem: name for name in image_names}
    labels[task] = {Path(name).stem: name for name in label_names}

distribution = pd.DataFrame({
    task: Counter(stem.split('_')[0] for stem in files)
    for task, files in images.items()
}).fillna(0).astype(int).sort_index()

distribution.index.name = 'Video'

In [7]:
detection_labels = {}

for task, mapping, excluded, fields_per_row in (
    ('detection', DETECTION_MAP, DETECTION_SKIP, 5),
    ('tracking', TRACKING_MAP, TRACKING_SKIP, 7)
):
    instances = defaultdict(Counter)
    background = Counter()
    with ZipFile(TASKS[task]) as archive:
        for stem, member in tqdm(sorted(labels[task].items())):
            video = stem.split("_")[0]
            converted = []
            for line in archive.read(member).decode("utf-8-sig").splitlines():
                fields = line.split()
                if not fields:
                    continue
                if len(fields) != fields_per_row:
                    raise ValueError(f'Invalid label format: {member}')
                
                raw_id = int(fields[0])
                if raw_id in excluded:
                    continue
                
                cid = mapping[raw_id]
                instances[video][cid] += 1
                converted.append(' '.join([str(cid), *fields[1:]]))
            background[video] += not converted
            if task == 'detection':
                detection_labels[stem] = '\n'.join(converted) + ('\n' if converted else '')
    
     
    for cid, name in enumerate(CLASSES):
        distribution[f'{task}:{name}'] = [instances[v][cid] for v in distribution.index]
    distribution[f'{task}:background'] = [background[v] for v in distribution.index]

  0%|          | 0/36478 [00:00<?, ?it/s]

  0%|          | 0/10022 [00:00<?, ?it/s]

In [8]:
image_video = {
    image['id']: Path(image['file_name']).stem.split('_')[0]
    for image in coco['images']
}
positive_ids = {
    a['image_id'] for a in coco['annotations'] if a['category_id'] == overflow_id
}

positive = Counter(image_video[image_id] for image_id in positive_ids)

distribution['overflow:positive'] = [positive[v] for v in distribution.index]
distribution['overflow:background'] = (
    distribution['segmentation'] - distribution['overflow:positive']
)
display(distribution)

,detection,tracking,segmentation,detection:container_yellow,detection:container_green,detection:container_default,detection:container_blue,detection:container_oil,detection:container_battery,detection:container_biodegradable,...,tracking:container_yellow,tracking:container_green,tracking:container_default,tracking:container_blue,tracking:container_oil,tracking:container_battery,tracking:container_biodegradable,tracking:background,overflow:positive,overflow:background
Video,,,,,,,,,,,,,,,,,,,,,
0001,240,240,0,0,0,395,0,0,0,0,...,0,0,395,0,0,0,0,0,0,0
0002,50,0,50,0,0,47,0,0,0,0,...,0,0,0,0,0,0,0,0,50,0
0003,290,0,290,0,0,549,0,0,0,0,...,0,0,0,0,0,0,0,0,290,0
0004,280,280,0,0,0,280,0,0,0,0,...,0,0,280,0,0,0,0,0,0,0
0005,180,180,0,0,0,180,0,0,0,0,...,0,0,180,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0211,360,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0212,250,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0213,380,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
videos = distribution.index.to_numpy()
features = distribution.loc[:, distribution.sum().gt(0)]

totals = features.sum().to_numpy()
values = features.to_numpy()

required = ~features.columns.str.endswith(':background')
boundaries = np.rint(np.cumsum(RATIOS)[:-1] * len(videos)).astype(int)
rng = np.random.default_rng(SEED)

best = None

for _ in range(N_TRIALS):
    groups = np.split(rng.permutation(len(videos)), boundaries)
    counts = np.stack([values[group].sum(axis=0) for group in groups])
    
    if (counts[:, required] == 0).any():
        continue
    
    errors = np.abs(counts / totals - RATIOS[:, None])
    score = errors[:, :len(TASKS)].mean() + errors[:, len(TASKS):].mean()
    if best is None or score < best[0]:
        best = (float(score), groups)


splits = {
    split: sorted(videos[group].tolist())
    for split, group in zip(SPLIT_NAMES, best[1])
}
video_split = {video: split for split, group in splits.items() for video in group}

In [ ]:
manifest = {
    'seed': SEED, 'trials': N_TRIALS,
    'group_by': 'video_id', 'target_ratios': dict(zip(SPLIT_NAMES, RATIOS.tolist())),
    'classes': CLASSES, 'method': 'seeded_video_search',
    'score': best[0], 'splits': splits,
    'source_archives': {task: path.name for task, path in TASKS.items()}
}

saved_manifest = OUTPUT / MANIFEST.name
MANIFEST.parent.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
MANIFEST.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

saved_manifest.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
distribution['Split'] = [video_split[video] for video in distribution.index]

display(distribution.groupby('Split').sum().reindex(SPLIT_NAMES))
print('Generated manifest:', MANIFEST)

,detection,tracking,segmentation,detection:container_yellow,detection:container_green,detection:container_default,detection:container_blue,detection:container_oil,detection:container_battery,detection:container_biodegradable,...,tracking:container_yellow,tracking:container_green,tracking:container_default,tracking:container_blue,tracking:container_oil,tracking:container_battery,tracking:container_biodegradable,tracking:background,overflow:positive,overflow:background
Split,,,,,,,,,,,,,,,,,,,,,
train,26424,7118,5090,7382,7060,19325,7689,1442,1534,941,...,3704,3706,10610,3882,487,1156,257,6,2764,2326
val,5018,1480,1030,1351,1478,4850,1606,249,461,208,...,709,861,2722,910,249,41,208,0,629,401
test,5036,1424,1110,1475,1380,4171,1635,182,600,585,...,597,574,2530,687,154,97,58,0,804,306


Generated manifest: /content/drive/MyDrive/WasteScope/manifests/video_split_7class_v2.json
